In [16]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
import yfinance as yf

In [17]:
import os
os.chdir('C:/Users/basleal/Desktop/tutorial/kaim-week-11')

In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
ticker = ['TSLA','BND','SPY']
starttime = '2015-01-01'
endtime = '2025-01-31'
data = yf.download(ticker,start=starttime,end=endtime)

[*********************100%***********************]  3 of 3 completed


Now we preprocess our data for our model

In [20]:
split_ratio=0.8
split_index = int(len(data)*split_ratio)
    #train_test
train = data.iloc[:split_index]
test = data.iloc[split_index:]

In [9]:
train.head()

Price           Close                              High              \
Ticker            BND         SPY       TSLA        BND         SPY   
Date                                                                  
2015-01-02  62.387100  172.592850  14.620667  62.417294  173.811083   
2015-01-05  62.568233  169.475891  14.006000  62.590878  171.702294   
2015-01-06  62.749432  167.879654  14.085333  62.938140  170.316096   
2015-01-07  62.787151  169.971634  14.063333  62.862634  170.316100   
2015-01-08  62.689003  172.987717  14.041333  62.734291  173.206165   

Price                        Low                              Open  \
Ticker           TSLA        BND         SPY       TSLA        BND   
Date                                                                 
2015-01-02  14.883333  62.213486  171.542657  14.217333  62.221036   
2015-01-05  14.433333  62.424813  169.165038  13.810667  62.455007   
2015-01-06  14.280000  62.673949  167.073100  13.614000  62.673949   
2015-01-07  14.318667  62.689025  168.770219  13.985333  62.756957   
2015-01-08  14.253333  62.628615  171.383032  14.000667  62.734291   

Price                               Volume                       
Ticker             SPY       TSLA      BND        SPY      TSLA  
Date                                                             
2015-01-02  173.391006  14.858000  2218800  121465900  71466000  
2015-01-05  171.534266  14.303333  5820100  169632600  80527500  
2015-01-06  169.786795  14.004000  3887600  209151400  93928500  
2015-01-07  169.223897  14.223333  2433400  125346700  44526000  
2015-01-08  171.399826  14.187333  1873400  147217800  51637500

In [21]:
train.columns=['_'.join(col) for col in train.columns]#flatten multiIndex
train.columns
test.columns=['_'.join(col) for col in test.columns]#flatten multiIndex using join

In [22]:
test.columns

Index(['Close_BND', 'Close_SPY', 'Close_TSLA', 'High_BND', 'High_SPY',
       'High_TSLA', 'Low_BND', 'Low_SPY', 'Low_TSLA', 'Open_BND', 'Open_SPY',
       'Open_TSLA', 'Volume_BND', 'Volume_SPY', 'Volume_TSLA'],
      dtype='object')

In [12]:
lstm_model = load_model('lstm.keras')
lstm_model.summary()

c:\Users\basleal\Desktop\tutorial\kaim-week-11\venv\Lib\site-packages\keras\src\saving\saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 12 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_16 (LSTM)                  │ (None, 1, 50)          │        10,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 1, 50)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_17 (LSTM)                  │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 25)             │         1,275 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 63,804 (249.24 KB)

 Trainable params: 31,901 (124.61 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 31,903 (124.62 KB)

Now we forecast for our datas on bnd and spy

In [23]:
X_bnd_train = train['Close_BND'].values.reshape(-1, 1)  # Ensure it's 2D
y_bnd_train = train['Close_BND'].values  # Target variable

X_bnd_test = test['Close_BND'].values.reshape(-1,1)
y_bnd_test = test['Close_BND'].values
X_spy_train = train['Close_SPY'].values.reshape(-1, 1)  # Ensure it's 2D
y_spy_train = train['Close_SPY'].values  # Target variable

X_spy_test = test['Close_SPY'].values.reshape(-1,1)
y_spy_test = test['Close_SPY'].values

In [33]:
from sklearn.preprocessing import MinMaxScaler

scaler_X = MinMaxScaler(feature_range=(0,1))
scaler_y = MinMaxScaler(feature_range=(0,1))

# Fit and transform X
X_train_bnd_scaled = scaler_X.fit_transform(X_bnd_train)
X_test_bnd_scaled = scaler_X.transform(X_bnd_test)

X_train_bnd_scaled = X_train_bnd_scaled.reshape((X_train_bnd_scaled.shape[0], 1, X_train_bnd_scaled.shape[1]))  
X_test_bnd_scaled = X_test_bnd_scaled.reshape((X_test_bnd_scaled.shape[0], 1, X_test_bnd_scaled.shape[1]))  

# Fit and transform y separately
y_train_bnd_scaled = scaler_y.fit_transform(y_bnd_train.reshape(-1, 1))
y_test_bnd_scaled = scaler_y.transform(y_bnd_test.reshape(-1, 1))

In [30]:
X_train_spy_scaled = scaler.fit_transform(X_bnd_train)
X_test_spy_scaled = scaler.transform(X_bnd_test)

X_train_spy_scaled = X_spy_train.reshape((X_spy_train.shape[0], 1, X_spy_train.shape[1]))  
X_test_spy_scaled = X_spy_test.reshape((X_spy_test.shape[0], 1, X_spy_test.shape[1]))  
y_train_spy_scaled = scaler.fit_transform(y_spy_train.reshape(-1, 1))
y_test_spy_scaled = scaler.transform(y_spy_test.reshape(-1, 1))

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0,1))
bnd_forecast = lstm_model.predict(X_test_bnd_scaled)

predicted_bnd = scaler_y.inverse_transform(bnd_forecast)
predicted_bnd

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step


NotFittedError: This MinMaxScaler instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [27]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0,1))
predicted_bnd = scaler.inverse_transform(bnd_forecast)
predicted_bnd

NotFittedError: This MinMaxScaler instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.